# Polygonal macro meshes

Local triangulation preserves the original macro faces. The research campaign uses the oscillatory PDE and P1/P3 degrees of the 2024 generalized RAD paper on five families: triangles, squares, rhombi, L-shaped cells and hexagons. Boundary-separated centroid fans satisfy the distinct-adjacent-triangle condition.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


## Declare the physical variational forms

The application defines local operators with the shared element, integration
and oriented face kernels and supplies the actual A/B/C blocks to the generic
assembler. For primal pressure the form is (K grad(p),grad(v)); mixed coordinates
use the inverse-permeability flux mass, divergence and normal-moment constraint.

$$
\begin{aligned}
A_Tu_T+B_T\lambda&=f_T,\\
C_Tu_T+D_T\lambda&=g_T, & C_T&=-B_T^{\mathsf T}, & D_T&=0.
\end{aligned}
$$

The local constant pressure and its physical integral moment retain the executed
nodal/modal basis. The separate global `Equation` supplies the signed pressure
boundary functional; pure Neumann data add a physical pressure integral constraint.
The flux trace follows fixed macroface normals. Mixed flux coordinates retain
their H(div) normal/interior moments and Piola transform; a primal gradient
field is evaluated independently. Generic assembly/solve and physical coefficient
interpretation are separate operations.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.darcy import define_darcy, recover_darcy

import numpy as np
from pymhm import PolygonMesh

mesh = PolygonMesh(
    np.array([[0, 0], [1, 0], [1, 0.5], [0.5, 0.5], [0.5, 1], [0, 1], [1, 1]]),
    (np.array([0, 1, 2, 3, 4, 5]), np.array([3, 2, 6, 4])),
    local_triangulation="boundary",
)
exact = lambda x: 1 + x[:, 0] + 2 * x[:, 1]
solution_definition = define_darcy(mesh, degree=2, dirichlet=exact, local_refinement=2)
solution_system = assemble(solution_definition.problem)
solution = recover_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
assert solution.l2_error(exact) < 1e-11
solution.l2_error(exact)


The full research campaign is acquired separately from this lightweight notebook. The following cells display its numerical record and publication figures.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display
record = json.loads((root / "examples/results/polygons.json").read_text())
record


In [ ]:
display(Image(filename=str(root / "docs/figures/polygons/convergence.png")))
display(Image(filename=str(root / "docs/figures/polygons/fields.png")))
